In [9]:
# =========================
# 1. Imports
# =========================
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    BertForMaskedLM,
    BertTokenizerFast
)

import kagglehub

In [10]:
# =========================
# 2. Download dataset
# =========================
path = kagglehub.dataset_download("rtatman/natural-stories-corpus")

print("Dataset path:", path)
print("Files:", os.listdir(path))


Using Colab cache for faster access to the 'natural-stories-corpus' dataset.
Dataset path: /kaggle/input/natural-stories-corpus
Files: ['words.tsv', 'batch2_pro.csv', 'batch1_pro.csv', 'all_stories.tok']


In [11]:
# =========================
# 3. File paths (FIXED)
# =========================
base_path = path

words_path = f"{base_path}/words.tsv"
batch1_path = f"{base_path}/batch1_pro.csv"
batch2_path = f"{base_path}/batch2_pro.csv"

# =========================
# 4. Load words.tsv
# =========================
df_words = pd.read_csv(
    words_path,
    sep="\t",
    names=["token_id", "token"]
)

def parse_token_id(token_id):
    parts = token_id.split(".")
    return int(parts[0]), int(parts[1]), parts[2]

df_words[["story_id", "position", "token_type"]] = df_words["token_id"].apply(
    lambda x: pd.Series(parse_token_id(x))
)

# Keep only actual words
df_words_clean = df_words[df_words["token_type"] == "word"].copy()
df_words_clean = df_words_clean.sort_values(["story_id", "position"])
df_words_clean.reset_index(drop=True, inplace=True)

# =========================
# 4B. Build chunk ids for safe word-level processing
# =========================
df_words_clean = df_words_clean.sort_values(["story_id", "position"]).copy()

CHUNK_SIZE = 200

df_words_clean["chunk_id"] = (
    df_words_clean.groupby("story_id").cumcount() // CHUNK_SIZE
)

print(df_words_clean.head())

# =========================
# 5. Load RT data
# =========================
batch1 = pd.read_csv(batch1_path)
batch2 = pd.read_csv(batch2_path)

df_rt = pd.concat([batch1, batch2], ignore_index=True)

df_rt = df_rt.rename(columns={
    "WorkerId": "subject",
    "item": "story_id",
    "zone": "position",
    "RT": "rt"
})

# =========================
# 6. Merge
# =========================
df_all = df_words_clean.merge(
    df_rt,
    on=["story_id", "position"]
)

print("Total merged rows:", len(df_all))

df = df_all.copy()

print("Subset rows:", len(df))

   token_id    token  story_id  position token_type  chunk_id
0  1.1.word       If         1         1       word         0
1  1.2.word      you         1         2       word         0
2  1.3.word     were         1         3       word         0
3  1.4.word       to         1         4       word         0
4  1.5.word  journey         1         5       word         0
Total merged rows: 1013377
Subset rows: 1013377


In [12]:
# =========================
# 7. Device
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================
# 8. Load models
# =========================
#GPT-2
gpt2_tokenizer = GPT2TokenizerFast.from_pretrained("gpt2", add_prefix_space=True)
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2")
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2_model.to(device)
gpt2_model.eval()

# BERT
bert_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
bert_model = BertForMaskedLM.from_pretrained("bert-base-uncased")
bert_model.to(device)
bert_model.eval()

torch.set_grad_enabled(False)

Using device: cuda


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [13]:
# Leave models in full precision for numerical stability.
# If runtime becomes a serious issue, you can re-enable .half() on GPU.
# if device.type == "cuda":
#     gpt2_model.half()
#     bert_model.half()

# =========================
# 9. GPT-2 word surprisal
# =========================
def compute_gpt2_word_surprisal(
    words,
    tokenizer,
    model,
    max_length=1024,
    stride=512,
    device="cpu"
):
    enc = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=False
    )

    input_ids = enc["input_ids"][0]
    word_ids = enc.word_ids()
    n_tokens = input_ids.size(0)

    token_surprisal = [None] * n_tokens
    start = 0

    while start < n_tokens:
        end = min(start + max_length, n_tokens)
        chunk_ids = input_ids[start:end].unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(chunk_ids).logits

        log_probs = F.log_softmax(logits, dim=-1)
        chunk_len = end - start

        # First token in a chunk cannot be scored unless it has a previous token
        if start == 0:
            score_from = 1
        else:
            overlap = max_length - stride
            score_from = overlap

        for local_idx in range(score_from, chunk_len):
            global_idx = start + local_idx
            true_token_id = chunk_ids[0, local_idx]
            pred_log_prob = log_probs[0, local_idx - 1, true_token_id].item()
            token_surprisal[global_idx] = -pred_log_prob

        if end == n_tokens:
            break
        start += stride

    word_to_surprisal = {}
    for tok_idx, wid in enumerate(word_ids):
        if wid is None:
            continue
        s = token_surprisal[tok_idx]
        if s is None:
            continue
        word_to_surprisal.setdefault(wid, 0.0)
        word_to_surprisal[wid] += s

    word_surprisal = []
    for wid in range(len(words)):
        word_surprisal.append(word_to_surprisal.get(wid, float("nan")))

    return word_surprisal

In [14]:
# =========================
# 10. BERT word pseudo-surprisal
# =========================
def compute_bert_word_pseudosurprisal(words, tokenizer, model, device="cpu"):
    enc = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        add_special_tokens=True,
        truncation=False
    )

    input_ids = enc["input_ids"][0]
    attention_mask = enc["attention_mask"][0]
    word_ids = enc.word_ids()

    word_to_token_positions = {}
    for tok_idx, wid in enumerate(word_ids):
        if wid is None:
            continue
        word_to_token_positions.setdefault(wid, []).append(tok_idx)

    word_surprisal = []

    for wid in range(len(words)):
        token_positions = word_to_token_positions.get(wid, [])
        if not token_positions:
            word_surprisal.append(float("nan"))
            continue

        word_log_prob = 0.0

        for pos in token_positions:
            masked_input = input_ids.clone()
            true_id = masked_input[pos].item()
            masked_input[pos] = tokenizer.mask_token_id

            with torch.no_grad():
                outputs = model(
                    input_ids=masked_input.unsqueeze(0).to(device),
                    attention_mask=attention_mask.unsqueeze(0).to(device)
                )
                logits = outputs.logits
                log_probs = F.log_softmax(logits, dim=-1)

            word_log_prob += log_probs[0, pos, true_id].item()

        word_surprisal.append(-word_log_prob)

    return word_surprisal

In [15]:
# =========================
# 11. Compute surprisals by story/chunk
# =========================
df_for_surprisal = df_words_clean.copy()
gpt2_results = []

for story_id, story_group in tqdm(df_words_clean.groupby("story_id", sort=True), desc="GPT-2 by story"):
    story_group = story_group.sort_values("position")
    words = story_group["token"].tolist()

    gpt2_vals = compute_gpt2_word_surprisal(
        words, gpt2_tokenizer, gpt2_model, device=device
    )

    if len(gpt2_vals) != len(words):
        raise ValueError(
            f"GPT-2 length mismatch in story {story_id}: {len(gpt2_vals)} vs {len(words)}"
        )

    out = story_group[["story_id", "position", "token"]].copy()
    out = out.rename(columns={"position": "word_position", "token": "word"})
    out["surprisal_gpt2"] = gpt2_vals
    gpt2_results.append(out)

gpt2_df = pd.concat(gpt2_results, ignore_index=True)
print(gpt2_df.head())

bert_results = []

for story_id, story_group in tqdm(df_words_clean.groupby("story_id", sort=True), desc="BERT by chunk"):
    for chunk_id, chunk_group in story_group.groupby("chunk_id", sort=True):
        chunk_group = chunk_group.sort_values("position")
        words = chunk_group["token"].tolist()

        if len(words) == 0:
            continue

        bert_test_enc = bert_tokenizer(
            words,
            is_split_into_words=True,
            add_special_tokens=True,
            truncation=False
        )

        if len(bert_test_enc["input_ids"]) > 512:
            print(f"Skipping story {story_id}, chunk {chunk_id}: too long for BERT")
            continue

        bert_vals = compute_bert_word_pseudosurprisal(
            words, bert_tokenizer, bert_model, device=device
        )

        if len(bert_vals) != len(words):
            raise ValueError(
                f"BERT length mismatch in story {story_id}, chunk {chunk_id}: "
                f"{len(bert_vals)} vs {len(words)}"
            )

        out = chunk_group[["story_id", "position", "token"]].copy()
        out = out.rename(columns={"position": "word_position", "token": "word"})
        out["surprisal_bert"] = bert_vals
        bert_results.append(out)

bert_df = pd.concat(bert_results, ignore_index=True)
print(bert_df.head())

final_df = gpt2_df.merge(
    bert_df,
    on=["story_id", "word_position", "word"],
    how="inner"
)

print(final_df.head())
print("Missing GPT-2:", final_df["surprisal_gpt2"].isna().sum())
print("Missing BERT:", final_df["surprisal_bert"].isna().sum())

GPT-2 by story:   0%|          | 0/10 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1172 > 1024). Running this sequence through the model will result in indexing errors


   story_id  word_position     word  surprisal_gpt2
0         1              1       If             NaN
1         1              2      you        3.345316
2         1              3     were        4.257098
3         1              4       to        1.615091
4         1              5  journey        9.814778


BERT by chunk:   0%|          | 0/10 [00:00<?, ?it/s]

   story_id  word_position     word  surprisal_bert
0         1              1       If        0.004285
1         1              2      you        0.001692
2         1              3     were        0.022344
3         1              4       to        0.002138
4         1              5  journey        6.463637
   story_id  word_position     word  surprisal_gpt2  surprisal_bert
0         1              1       If             NaN        0.004285
1         1              2      you        3.345316        0.001692
2         1              3     were        4.257098        0.022344
3         1              4       to        1.615091        0.002138
4         1              5  journey        9.814778        6.463637
Missing GPT-2: 8
Missing BERT: 0


In [17]:
# =========================
# 12. Merge surprisal table back into RT dataframe
# =========================
df_final = df_all.merge(
    final_df,
    left_on=["story_id", "position", "token"],
    right_on=["story_id", "word_position", "word"],
    how="left"
)

print("Merged rows:", len(df_final))
print("Missing GPT-2 surprisals:", df_final["surprisal_gpt2"].isna().sum())
print("Missing BERT surprisals:", df_final["surprisal_bert"].isna().sum())
df_final.head()

Merged rows: 1013377
Missing GPT-2 surprisals: 793
Missing BERT surprisals: 0


,token_id,token,story_id,position,token_type,chunk_id,subject,WorkTimeInSeconds,correct,rt,word_position,word,surprisal_gpt2,surprisal_bert
0,1.1.word,If,1,1,word,0,A37I1ETWW49IZO,5602,4,3061,1,If,NaN,0.004285
1,1.1.word,If,1,1,word,0,A3QJPB0NZU5PY1,3960,6,1527,1,If,NaN,0.004285
2,1.1.word,If,1,1,word,0,A2RPQGUWVZPX7U,2431,5,1272,1,If,NaN,0.004285
3,1.1.word,If,1,1,word,0,A11KMPAZSE5Q0Q,1287,5,1002,1,If,NaN,0.004285
4,1.1.word,If,1,1,word,0,A1U1QL617G5DU3,2074,6,1404,1,If,NaN,0.004285


In [18]:
# =========================
# 13. Validation + Export
# =========================
def validate_surprisal_df(df_out):
    required_cols = [
        "story_id", "position", "token",
        "surprisal_gpt2", "surprisal_bert"
    ]

    missing = [c for c in required_cols if c not in df_out.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    dup_count = df_out.duplicated(subset=["subject", "story_id", "position"]).sum()
    if dup_count > 0:
        print(f"Warning: found {dup_count} duplicated subject/story/position rows")

    missing_gpt2 = df_out["surprisal_gpt2"].isna().sum()
    missing_bert = df_out["surprisal_bert"].isna().sum()

    print("Rows:", len(df_out))
    print("Missing GPT-2:", missing_gpt2)
    print("Missing BERT:", missing_bert)
    print(df_out[["surprisal_gpt2", "surprisal_bert"]].describe())

    if missing_gpt2 > 0 or missing_bert > 0:
        print("\nSample rows with missing surprisals:")
        print(
            df_out.loc[
                df_out["surprisal_gpt2"].isna() | df_out["surprisal_bert"].isna(),
                ["story_id", "position", "token", "surprisal_gpt2", "surprisal_bert"]
            ].head(10).to_string(index=False)
        )

validate_surprisal_df(df_final)

output_path = "df_final_with_surprisals.csv"
df_final.to_csv(output_path, index=False)
print(f"DataFrame saved to {output_path}")

Rows: 1013377
Missing GPT-2: 793
Missing BERT: 0
       surprisal_gpt2  surprisal_bert
count    1.012584e+06    1.013377e+06
mean     3.967432e+00    2.072838e+00
std      3.219317e+00    2.930645e+00
min      4.515820e-04    3.695481e-06
25%      1.547008e+00    5.868993e-02
50%      3.323409e+00    6.707137e-01
75%      5.630807e+00    3.234190e+00
max      4.095464e+01    2.895910e+01

Sample rows with missing surprisals:
 story_id  position token  surprisal_gpt2  surprisal_bert
        1         1    If             NaN        0.004285
        1         1    If             NaN        0.004285
        1         1    If             NaN        0.004285
        1         1    If             NaN        0.004285
        1         1    If             NaN        0.004285
        1         1    If             NaN        0.004285
        1         1    If             NaN        0.004285
        1         1    If             NaN        0.004285
        1         1    If             NaN        

In [20]:
# =========================
# 14. Sanity check to inspect a few rows
# =========================
sample_check = df_final.sample(20, random_state=42)[
    ["story_id", "chunk_id", "word_position", "word", "surprisal_gpt2", "surprisal_bert"]
].sort_values(["story_id", "chunk_id", "word_position"])

print(sample_check.to_string(index=False))

 story_id  chunk_id  word_position       word  surprisal_gpt2  surprisal_bert
        2         0             80       know        2.449125        1.505544
        2         1            221        had        3.739695        0.128592
        2         3            682      drops        0.584099        0.158039
        2         4            878         to        0.011604        0.000521
        3         4            910      there        3.119569        0.101165
        5         0             29     seemed        7.295686        0.908759
        5         0             81       nine        3.323854        2.473590
        5         2            529          I        3.482445        0.150763
        6         5           1011       weed        1.214831        7.707985
        7         3            671 microphone        7.064418        4.697315
        7         3            718         up        0.132868        0.165044
        7         4            808        her        3.631686   